# Neural Machine Translation using Transformer Architecture — Unified Synthetic + Real Pipeline

This notebook is rebuilt to use synthetic translation pairs first, then real public translation data, through the same translation/evaluation pipeline. Streamlit export is placed at the last section.

In [1]:
# Cell 001: Project banner
PROJECT_NAME = "NMT Transformer Unified Synthetic-to-Real Translation System"
print(PROJECT_NAME)

NMT Transformer Unified Synthetic-to-Real Translation System


In [2]:
# Cell 002: Import core libraries
import os, re, io, json, zipfile, math, random, time, ast
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime
from typing import Any, Dict, List, Tuple, Optional
from collections import Counter, defaultdict

In [3]:
# Cell 003: Import data science libraries
import numpy as np
import pandas as pd

In [4]:
# Cell 004: Optional dataset loader
try:
    from datasets import load_dataset
    HAS_DATASETS = True
except Exception as exc:
    load_dataset = None
    HAS_DATASETS = False
    print("datasets not available:", exc)

In [5]:
# Cell 005: Optional torch / transformer readiness
try:
    import torch
    TORCH_AVAILABLE = True
except Exception as exc:
    torch = None
    TORCH_AVAILABLE = False
    print("torch not available:", exc)

In [6]:
# Cell 006: Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if TORCH_AVAILABLE:
    torch.manual_seed(SEED)
print("Seed:", SEED)

Seed: 42


In [7]:
# Cell 007: Configuration dictionary
CONFIG = {
    "source_lang": "en",
    "target_lang": "fr",
    "synthetic_pairs": 120,
    "real_pairs": 200,
    "output_root": "outputs",
    "project_slug": "nmt_transformer_unified",
}
CONFIG

{'source_lang': 'en',
 'target_lang': 'fr',
 'synthetic_pairs': 120,
 'real_pairs': 200,
 'output_root': 'outputs',
 'project_slug': 'nmt_transformer_unified'}

In [8]:
# Cell 008: Create versioned output directory
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(CONFIG["output_root"]) / f"{CONFIG['project_slug']}_{RUN_ID}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR.resolve())

Output directory: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\Transformer\Neural Machine Translation using Transformer\outputs\nmt_transformer_unified_20260428_151647


In [9]:
# Cell 009: Notebook-level utility display function
def show_df(df: pd.DataFrame, n: int = 5):
    display(df.head(n))
    print("shape:", df.shape)

In [10]:
# Cell 010: Normalize text utility
# This cell is intentionally global because many later cells export CSV/Excel.
# Public datasets can contain hidden ASCII control characters such as \x01,
# which openpyxl refuses to write into .xlsx files.

EXCEL_ILLEGAL_CHAR_PATTERN = re.compile(r"[\x00-\x08\x0B-\x0C\x0E-\x1F]")


def normalize_text(text: Any) -> str:
    """Normalize text used by the translation pipeline."""
    if text is None:
        return ""
    try:
        if isinstance(text, float) and pd.isna(text):
            return ""
    except Exception:
        pass

    text = str(text).replace("\xa0", " ")
    text = EXCEL_ILLEGAL_CHAR_PATTERN.sub(" ", text)
    text = text.replace("\ufffe", " ").replace("\uffff", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def sanitize_for_excel(value: Any) -> Any:
    """Make a single value safe for openpyxl Excel export."""
    if value is None:
        return ""
    try:
        if isinstance(value, float) and pd.isna(value):
            return ""
    except Exception:
        pass

    if isinstance(value, (list, tuple, dict)):
        value = json.dumps(value, ensure_ascii=False)

    if isinstance(value, str):
        value = value.replace("\xa0", " ")
        value = EXCEL_ILLEGAL_CHAR_PATTERN.sub(" ", value)
        value = value.replace("\ufffe", " ").replace("\uffff", " ")
        value = re.sub(r"\s+", " ", value).strip()
        return value[:32000]

    return value


def sanitize_dataframe_for_excel(df: pd.DataFrame) -> pd.DataFrame:
    """Return a fully sanitized dataframe safe for openpyxl."""
    if df is None:
        return pd.DataFrame()

    clean = df.copy()
    clean.columns = [sanitize_for_excel(str(c))[:31] if c is not None else "" for c in clean.columns]

    try:
        clean = clean.map(sanitize_for_excel)
    except AttributeError:
        clean = clean.applymap(sanitize_for_excel)

    return clean


In [11]:
# Cell 011: Tokenizer utility
def tokenize(text: Any) -> List[str]:
    return re.findall(r"[A-Za-zÀ-ÿ0-9']+|[.,!?;:]", normalize_text(text).lower())

In [12]:
# Cell 012: Detokenizer utility
def detokenize(tokens: List[str]) -> str:
    text = " ".join(tokens)
    text = re.sub(r"\s+([.,!?;:])", r"", text)
    return text.strip()

In [13]:
# Cell 013: Translation pair schema
@dataclass
class TranslationPair:
    pair_id: str
    source_text: str
    target_text: str
    source_lang: str
    target_lang: str
    source_type: str

In [14]:
# Cell 014: Convert translation pairs to dataframe
def pairs_to_df(pairs: List[TranslationPair]) -> pd.DataFrame:
    return pd.DataFrame([asdict(p) for p in pairs])

In [15]:
# Cell 015: Basic quality validator
def validate_pairs(pairs: List[TranslationPair], label: str) -> Dict[str, Any]:
    non_empty = [p for p in pairs if normalize_text(p.source_text) and normalize_text(p.target_text)]
    return {"label": label, "pairs": len(pairs), "non_empty_pairs": len(non_empty), "valid": len(non_empty) > 0}

In [16]:
# Cell 016: Synthetic vocabulary components
SYN_SUBJECTS = [("the engineer", "l'ingénieur"), ("the model", "le modèle"), ("the system", "le système"), ("the dashboard", "le tableau de bord"), ("the pipeline", "le pipeline")]
SYN_VERBS = [("improves", "améliore"), ("checks", "vérifie"), ("analyzes", "analyse"), ("predicts", "prédit"), ("summarizes", "résume")]
SYN_OBJECTS = [("quality", "la qualité"), ("data", "les données"), ("results", "les résultats"), ("performance", "la performance"), ("risk", "le risque")]
print(len(SYN_SUBJECTS), len(SYN_VERBS), len(SYN_OBJECTS))

5 5 5


In [17]:
# Cell 017: Synthetic translation pair generator
def make_synthetic_pairs(n: int = 120) -> List[TranslationPair]:
    rows = []
    idx = 0
    for s_en, s_fr in SYN_SUBJECTS:
        for v_en, v_fr in SYN_VERBS:
            for o_en, o_fr in SYN_OBJECTS:
                rows.append(TranslationPair(
                    pair_id=f"syn_{idx:04d}",
                    source_text=f"{s_en} {v_en} {o_en}.",
                    target_text=f"{s_fr} {v_fr} {o_fr}.",
                    source_lang="en",
                    target_lang="fr",
                    source_type="synthetic",
                ))
                idx += 1
    return rows[:n]

In [18]:
# Cell 018: Build synthetic dataset first
synthetic_pairs = make_synthetic_pairs(CONFIG["synthetic_pairs"])
synthetic_df = pairs_to_df(synthetic_pairs)
show_df(synthetic_df, 5)

,pair_id,source_text,target_text,source_lang,target_lang,source_type
0,syn_0000,the engineer improves quality.,l'ingénieur améliore la qualité.,en,fr,synthetic
1,syn_0001,the engineer improves data.,l'ingénieur améliore les données.,en,fr,synthetic
2,syn_0002,the engineer improves results.,l'ingénieur améliore les résultats.,en,fr,synthetic
3,syn_0003,the engineer improves performance.,l'ingénieur améliore la performance.,en,fr,synthetic
4,syn_0004,the engineer improves risk.,l'ingénieur améliore le risque.,en,fr,synthetic


shape: (120, 6)


In [19]:
# Cell 019: Validate synthetic dataset
synthetic_validation = validate_pairs(synthetic_pairs, "synthetic")
synthetic_validation

{'label': 'synthetic', 'pairs': 120, 'non_empty_pairs': 120, 'valid': True}

In [20]:
# Cell 020: Synthetic source token stats
synthetic_source_tokens = Counter(t for p in synthetic_pairs for t in tokenize(p.source_text))
synthetic_source_tokens.most_common(10)

[('the', 120),
 ('.', 120),
 ('engineer', 25),
 ('improves', 25),
 ('checks', 25),
 ('analyzes', 25),
 ('predicts', 25),
 ('model', 25),
 ('system', 25),
 ('dashboard', 25)]

In [21]:
# Cell 021: Synthetic target token stats
synthetic_target_tokens = Counter(t for p in synthetic_pairs for t in tokenize(p.target_text))
synthetic_target_tokens.most_common(10)

[('.', 120),
 ('le', 119),
 ('la', 48),
 ('les', 48),
 ("l'ingénieur", 25),
 ('améliore', 25),
 ('vérifie', 25),
 ('analyse', 25),
 ('prédit', 25),
 ('modèle', 25)]

In [22]:
# Cell 022: Train/test split helper
def split_pairs(pairs: List[TranslationPair], train_ratio: float = 0.8):
    pairs = list(pairs)
    random.Random(SEED).shuffle(pairs)
    cut = max(1, int(len(pairs) * train_ratio))
    return pairs[:cut], pairs[cut:]

In [23]:
# Cell 023: Split synthetic pairs
synthetic_train_pairs, synthetic_test_pairs = split_pairs(synthetic_pairs)
len(synthetic_train_pairs), len(synthetic_test_pairs)

(96, 24)

In [24]:
# Cell 024: Simple vocabulary builder
class SimpleVocab:
    def __init__(self, min_freq: int = 1):
        self.min_freq = min_freq
        self.token_to_id = {"<pad>": 0, "<unk>": 1, "<bos>": 2, "<eos>": 3}
        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
    def fit(self, texts: List[str]):
        counts = Counter(t for text in texts for t in tokenize(text))
        for tok, cnt in counts.items():
            if cnt >= self.min_freq and tok not in self.token_to_id:
                idx = len(self.token_to_id)
                self.token_to_id[tok] = idx
                self.id_to_token[idx] = tok
        return self
    def encode(self, text: str, add_special: bool = True) -> List[int]:
        ids = [self.token_to_id.get(t, self.token_to_id["<unk>"]) for t in tokenize(text)]
        return [2] + ids + [3] if add_special else ids
    def decode(self, ids: List[int]) -> str:
        toks = [self.id_to_token.get(int(i), "<unk>") for i in ids if int(i) not in (0,2,3)]
        return detokenize(toks)
    def __len__(self):
        return len(self.token_to_id)

In [25]:
# Cell 025: Fit vocabularies on synthetic data
src_vocab = SimpleVocab().fit([p.source_text for p in synthetic_train_pairs])
tgt_vocab = SimpleVocab().fit([p.target_text for p in synthetic_train_pairs])
len(src_vocab), len(tgt_vocab)

(21, 25)

In [26]:
# Cell 026: Encode one synthetic example
sample_pair = synthetic_pairs[0]
src_vocab.encode(sample_pair.source_text), tgt_vocab.encode(sample_pair.target_text)

([2, 4, 5, 19, 11, 8, 3], [2, 4, 23, 13, 14, 8, 3])

In [27]:
# Cell 027: Lightweight translator baseline
class DictionaryTranslator:
    def __init__(self):
        self.phrase_table = {}
        self.word_table = {}
        self.trained_pairs = []
    def fit(self, pairs: List[TranslationPair]):
        self.trained_pairs = list(pairs)
        self.phrase_table = {normalize_text(p.source_text).lower(): normalize_text(p.target_text) for p in pairs}
        align_counts = defaultdict(Counter)
        for p in pairs:
            src = [t for t in tokenize(p.source_text) if re.search(r"[a-zA-Z]", t)]
            tgt = [t for t in tokenize(p.target_text) if re.search(r"[a-zA-ZÀ-ÿ]", t)]
            for s, t in zip(src, tgt):
                align_counts[s][t] += 1
        self.word_table = {s: c.most_common(1)[0][0] for s, c in align_counts.items() if c}
        return self
    def translate(self, text: str) -> str:
        key = normalize_text(text).lower()
        if key in self.phrase_table:
            return self.phrase_table[key]
        out = []
        for tok in tokenize(text):
            if tok in ".,!?;:":
                out.append(tok)
            else:
                out.append(self.word_table.get(tok.lower(), tok))
        return detokenize(out)

In [28]:
# Cell 028: Train synthetic-only translator
synthetic_translator = DictionaryTranslator().fit(synthetic_train_pairs)
print(synthetic_translator.translate("the model analyzes data."))

le modèle analyse les données.


In [29]:
# Cell 029: Unigram F1 metric
def unigram_f1(pred: str, gold: str) -> float:
    p = Counter(tokenize(pred)); g = Counter(tokenize(gold))
    if not p or not g:
        return 0.0
    overlap = sum((p & g).values())
    precision = overlap / max(sum(p.values()), 1)
    recall = overlap / max(sum(g.values()), 1)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

In [30]:
# Cell 030: Exact match metric
def exact_match(pred: str, gold: str) -> int:
    return int(normalize_text(pred).lower() == normalize_text(gold).lower())

In [31]:
# Cell 031: Evaluate translation pairs
def evaluate_pairs(model: DictionaryTranslator, pairs: List[TranslationPair], label: str) -> pd.DataFrame:
    rows = []
    for p in pairs:
        pred = model.translate(p.source_text)
        rows.append({
            "eval_label": label,
            "pair_id": p.pair_id,
            "source_type": p.source_type,
            "source_text": p.source_text,
            "gold_translation": p.target_text,
            "pred_translation": pred,
            "unigram_f1": unigram_f1(pred, p.target_text),
            "exact_match": exact_match(pred, p.target_text),
        })
    return pd.DataFrame(rows)

In [32]:
# Cell 032: Evaluate synthetic baseline
synthetic_eval_df = evaluate_pairs(synthetic_translator, synthetic_test_pairs, "synthetic_baseline")
show_df(synthetic_eval_df, 5)

,eval_label,pair_id,source_type,source_text,gold_translation,pred_translation,unigram_f1,exact_match
0,synthetic_baseline,syn_0102,synthetic,the pipeline improves results.,le pipeline améliore les résultats.,le pipeline améliore les,0.800000,0
1,synthetic_baseline,syn_0077,synthetic,the dashboard improves results.,le tableau de bord améliore les résultats.,le tableau améliore les,0.666667,0
2,synthetic_baseline,syn_0064,synthetic,the system analyzes risk.,le système analyse le risque.,le système analyse le,0.800000,0
3,synthetic_baseline,syn_0029,synthetic,the model improves risk.,le modèle améliore le risque.,le modèle améliore le,0.800000,0
4,synthetic_baseline,syn_0027,synthetic,the model improves results.,le modèle améliore les résultats.,le modèle améliore les,0.800000,0


shape: (24, 8)


In [33]:
# Cell 033: Summarize evaluation helper
def summarize_eval(df: pd.DataFrame, label: str) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame([{"label": label, "rows": 0, "mean_unigram_f1": np.nan, "exact_match_rate": np.nan}])
    return pd.DataFrame([{"label": label, "rows": len(df), "mean_unigram_f1": df["unigram_f1"].mean(), "exact_match_rate": df["exact_match"].mean()}])

In [34]:
# Cell 034: Synthetic baseline summary
synthetic_baseline_summary = summarize_eval(synthetic_eval_df, "synthetic_baseline")
synthetic_baseline_summary

,label,rows,mean_unigram_f1,exact_match_rate
0,synthetic_baseline,24,0.683333,0.0


In [35]:
# Cell 035: Real dataset loader attempts
REAL_DATASET_ATTEMPTS = [
    ("opus_books", "en-fr", "train"),
    ("Helsinki-NLP/opus_books", "en-fr", "train"),
]
REAL_DATASET_ATTEMPTS

[('opus_books', 'en-fr', 'train'),
 ('Helsinki-NLP/opus_books', 'en-fr', 'train')]

In [36]:
# Cell 036: Parse dataset translation row
def parse_translation_row(row: Dict[str, Any], idx: int, source_type: str) -> Optional[TranslationPair]:
    tr = row.get("translation", {}) if isinstance(row, dict) else {}
    en = normalize_text(tr.get("en", "")) if isinstance(tr, dict) else ""
    fr = normalize_text(tr.get("fr", "")) if isinstance(tr, dict) else ""
    if not en or not fr:
        return None
    return TranslationPair(f"real_{idx:04d}", en, fr, "en", "fr", source_type)

In [37]:
# Cell 037: Fallback public-domain-like translation sample
def fallback_real_pairs() -> List[TranslationPair]:
    rows = [
        ("I am reading a book.", "Je lis un livre."),
        ("The weather is beautiful today.", "Le temps est beau aujourd'hui."),
        ("We need more data.", "Nous avons besoin de plus de données."),
        ("This model translates text.", "Ce modèle traduit le texte."),
        ("The result is important.", "Le résultat est important."),
        ("She writes a letter.", "Elle écrit une lettre."),
        ("They are building a system.", "Ils construisent un système."),
        ("The team evaluates performance.", "L'équipe évalue la performance."),
    ]
    return [TranslationPair(f"real_fallback_{i:04d}", en, fr, "en", "fr", "real_fallback_public_sample") for i, (en, fr) in enumerate(rows)]

In [38]:
# Cell 038: Real public translation data loader
def load_real_pairs(max_pairs: int = 200) -> Tuple[List[TranslationPair], str]:
    if HAS_DATASETS:
        for dataset_name, subset, split in REAL_DATASET_ATTEMPTS:
            try:
                ds = load_dataset(dataset_name, subset, split=f"{split}[:{max_pairs}]")
                pairs = []
                for i, row in enumerate(ds):
                    p = parse_translation_row(row, i, "real_opus_books")
                    if p is not None:
                        pairs.append(p)
                if pairs:
                    print(f"Loaded real translation data from {dataset_name}/{subset}: {len(pairs)} pairs")
                    return pairs, f"{dataset_name}/{subset}"
            except Exception as exc:
                print(f"Could not load {dataset_name}/{subset}: {type(exc).__name__}: {exc}")
    pairs = fallback_real_pairs()
    print("Using fallback public sample pairs:", len(pairs))
    return pairs, "fallback_public_sample"

In [39]:
# Cell 039: Load real public translation pairs after synthetic phase
real_pairs, real_data_source = load_real_pairs(CONFIG["real_pairs"])
real_df = pairs_to_df(real_pairs)
print("Real source:", real_data_source)
show_df(real_df, 5)

Loaded real translation data from opus_books/en-fr: 200 pairs
Real source: opus_books/en-fr


,pair_id,source_text,target_text,source_lang,target_lang,source_type
0,real_0000,The Wanderer,Le grand Meaulnes,en,fr,real_opus_books
1,real_0001,Alain-Fournier,Alain-Fournier,en,fr,real_opus_books
2,real_0002,First Part,PREMIÈRE PARTIE,en,fr,real_opus_books
3,real_0003,I,CHAPITRE PREMIER,en,fr,real_opus_books
4,real_0004,THE BOARDER,LE PENSIONNAIRE,en,fr,real_opus_books


shape: (200, 6)


In [40]:
# Cell 040: Validate real dataset
real_validation = validate_pairs(real_pairs, "real")
real_validation

{'label': 'real', 'pairs': 200, 'non_empty_pairs': 200, 'valid': True}

In [41]:
# Cell 041: Real source type counts
real_df["source_type"].value_counts() if len(real_df) else pd.Series(dtype=int)

source_type
real_opus_books    200
Name: count, dtype: int64

In [42]:
# Cell 042: Split real pairs
real_train_pairs, real_test_pairs = split_pairs(real_pairs)
len(real_train_pairs), len(real_test_pairs)

(160, 40)

In [43]:
# Cell 043: Build unified synthetic + real corpus
unified_train_pairs = synthetic_train_pairs + real_train_pairs
unified_test_pairs = synthetic_test_pairs + real_test_pairs
len(unified_train_pairs), len(unified_test_pairs)

(256, 64)

In [44]:
# Cell 044: Fit unified vocabularies
unified_src_vocab = SimpleVocab().fit([p.source_text for p in unified_train_pairs])
unified_tgt_vocab = SimpleVocab().fit([p.target_text for p in unified_train_pairs])
len(unified_src_vocab), len(unified_tgt_vocab)

(1168, 1208)

In [45]:
# Cell 045: Train unified translator on synthetic + real
unified_translator = DictionaryTranslator().fit(unified_train_pairs)
print("Unified translator trained pairs:", len(unified_translator.trained_pairs))

Unified translator trained pairs: 256


In [46]:
# Cell 046: Evaluate unified translator on synthetic test set
synthetic_unified_eval_df = evaluate_pairs(unified_translator, synthetic_test_pairs, "synthetic_on_unified")
show_df(synthetic_unified_eval_df, 5)

,eval_label,pair_id,source_type,source_text,gold_translation,pred_translation,unigram_f1,exact_match
0,synthetic_on_unified,syn_0102,synthetic,the pipeline improves results.,le pipeline améliore les résultats.,le pipeline améliore les,0.800000,0
1,synthetic_on_unified,syn_0077,synthetic,the dashboard improves results.,le tableau de bord améliore les résultats.,le tableau améliore les,0.666667,0
2,synthetic_on_unified,syn_0064,synthetic,the system analyzes risk.,le système analyse le risque.,le système analyse le,0.800000,0
3,synthetic_on_unified,syn_0029,synthetic,the model improves risk.,le modèle améliore le risque.,le modèle améliore le,0.800000,0
4,synthetic_on_unified,syn_0027,synthetic,the model improves results.,le modèle améliore les résultats.,le modèle améliore les,0.800000,0


shape: (24, 8)


In [47]:
# Cell 047: Evaluate unified translator on real test set
real_unified_eval_df = evaluate_pairs(unified_translator, real_test_pairs, "real_on_unified")
show_df(real_unified_eval_df, 5)

,eval_label,pair_id,source_type,source_text,gold_translation,pred_translation,unigram_f1,exact_match
0,real_on_unified,real_0086,real_opus_books,"Then, taking hold of my hand, he quickly drew ...","Puis, me prenant par la main, il m’entraîna vi...",alors prenant hold de ma en il vite drew moi...,0.230769,0
1,real_on_unified,real_0181,real_opus_books,Both he and his man began to laugh.,"Et tous les deux, son ouvrier et lui, se prire...",rien il et sa ouvrier à de doucement,0.260870,0
2,real_on_unified,real_0039,real_opus_books,She was small and wore a black old- fashioned ...,"Elle était petite, coiffée d’une capote de vel...",elle était à et wore un black ancienne qui vel...,0.296296,0
3,real_on_unified,real_0168,real_opus_books,He held the piece of iron on which he had been...,"Il regardait, en l’approchant de son tablier d...",il suivit le la de la sur de il d instant avoi...,0.333333,0
4,real_on_unified,real_0087,real_opus_books,"A moment, later, as she came out of the door w...","Un instant après, ma mère qui sortait sur le p...",un à instant comme elle venait ne de le port...,0.373333,0


shape: (40, 8)


In [48]:
# Cell 048: Evaluate mixed test set
mixed_unified_eval_df = evaluate_pairs(unified_translator, unified_test_pairs, "mixed_on_unified")
show_df(mixed_unified_eval_df, 5)

,eval_label,pair_id,source_type,source_text,gold_translation,pred_translation,unigram_f1,exact_match
0,mixed_on_unified,syn_0102,synthetic,the pipeline improves results.,le pipeline améliore les résultats.,le pipeline améliore les,0.800000,0
1,mixed_on_unified,syn_0077,synthetic,the dashboard improves results.,le tableau de bord améliore les résultats.,le tableau améliore les,0.666667,0
2,mixed_on_unified,syn_0064,synthetic,the system analyzes risk.,le système analyse le risque.,le système analyse le,0.800000,0
3,mixed_on_unified,syn_0029,synthetic,the model improves risk.,le modèle améliore le risque.,le modèle améliore le,0.800000,0
4,mixed_on_unified,syn_0027,synthetic,the model improves results.,le modèle améliore les résultats.,le modèle améliore les,0.800000,0


shape: (64, 8)


In [49]:
# Cell 049: Combine all evaluation results
all_eval_df = pd.concat([synthetic_eval_df, synthetic_unified_eval_df, real_unified_eval_df, mixed_unified_eval_df], ignore_index=True)
show_df(all_eval_df, 8)

,eval_label,pair_id,source_type,source_text,gold_translation,pred_translation,unigram_f1,exact_match
0,synthetic_baseline,syn_0102,synthetic,the pipeline improves results.,le pipeline améliore les résultats.,le pipeline améliore les,0.800000,0
1,synthetic_baseline,syn_0077,synthetic,the dashboard improves results.,le tableau de bord améliore les résultats.,le tableau améliore les,0.666667,0
2,synthetic_baseline,syn_0064,synthetic,the system analyzes risk.,le système analyse le risque.,le système analyse le,0.800000,0
3,synthetic_baseline,syn_0029,synthetic,the model improves risk.,le modèle améliore le risque.,le modèle améliore le,0.800000,0
4,synthetic_baseline,syn_0027,synthetic,the model improves results.,le modèle améliore les résultats.,le modèle améliore les,0.800000,0
5,synthetic_baseline,syn_0106,synthetic,the pipeline checks data.,le pipeline vérifie les données.,le pipeline vérifie les,0.800000,0
6,synthetic_baseline,syn_0117,synthetic,the pipeline predicts results.,le pipeline prédit les résultats.,le pipeline prédit les,0.800000,0
7,synthetic_baseline,syn_0004,synthetic,the engineer improves risk.,l'ingénieur améliore le risque.,le vérifie améliore le,0.444444,0


shape: (152, 8)


In [50]:
# Cell 050: Evaluation summary table
eval_summary_df = pd.concat([
    summarize_eval(synthetic_eval_df, "synthetic_baseline"),
    summarize_eval(synthetic_unified_eval_df, "synthetic_on_unified"),
    summarize_eval(real_unified_eval_df, "real_on_unified"),
    summarize_eval(mixed_unified_eval_df, "mixed_on_unified"),
], ignore_index=True)
eval_summary_df

,label,rows,mean_unigram_f1,exact_match_rate
0,synthetic_baseline,24,0.683333,0.0
1,synthetic_on_unified,24,0.683333,0.0
2,real_on_unified,40,0.232904,0.0
3,mixed_on_unified,64,0.401815,0.0


In [51]:
# Cell 051: Save synthetic dataset preview
synthetic_df.to_csv(OUTPUT_DIR / "synthetic_translation_pairs.csv", index=False)
print("saved synthetic_translation_pairs.csv")

saved synthetic_translation_pairs.csv


In [52]:
# Cell 052: Save real dataset preview
real_df.to_csv(OUTPUT_DIR / "real_translation_pairs.csv", index=False)
print("saved real_translation_pairs.csv")

saved real_translation_pairs.csv


In [53]:
# Cell 053: Save evaluation results
all_eval_df.to_csv(OUTPUT_DIR / "translation_evaluation_results.csv", index=False)
print("saved translation_evaluation_results.csv")

saved translation_evaluation_results.csv


In [54]:
# Cell 054: Save summary results
eval_summary_df.to_csv(OUTPUT_DIR / "translation_summary.csv", index=False)
print("saved translation_summary.csv")

saved translation_summary.csv


In [55]:
# Cell 055: Source type distribution
source_distribution_df = pd.concat([synthetic_df, real_df], ignore_index=True)["source_type"].value_counts().rename_axis("source_type").reset_index(name="count")
source_distribution_df

,source_type,count
0,real_opus_books,200
1,synthetic,120


In [56]:
# Cell 056: Average source length by source type
combined_pairs_df = pd.concat([synthetic_df, real_df], ignore_index=True)
combined_pairs_df["source_len"] = combined_pairs_df["source_text"].apply(lambda x: len(tokenize(x)))
combined_pairs_df["target_len"] = combined_pairs_df["target_text"].apply(lambda x: len(tokenize(x)))
length_summary_df = combined_pairs_df.groupby("source_type")[["source_len", "target_len"]].mean().reset_index()
length_summary_df

,source_type,source_len,target_len
0,real_opus_books,24.505,24.915000
1,synthetic,5.000,6.208333


In [57]:
# Cell 057: Vocabulary coverage diagnostics
coverage_rows = []
for label, pairs in [("synthetic_test", synthetic_test_pairs), ("real_test", real_test_pairs)]:
    toks = [t for p in pairs for t in tokenize(p.source_text)]
    known = sum(t in unified_src_vocab.token_to_id for t in toks)
    coverage_rows.append({"label": label, "tokens": len(toks), "known": known, "coverage": known / max(len(toks), 1)})
coverage_df = pd.DataFrame(coverage_rows)
coverage_df

,label,tokens,known,coverage
0,synthetic_test,120,120,1.000000
1,real_test,884,731,0.826923


In [58]:
# Cell 058: Error analysis table
error_analysis_df = all_eval_df.sort_values("unigram_f1").head(20).copy()
error_analysis_df[["eval_label", "source_text", "gold_translation", "pred_translation", "unigram_f1"]]

,eval_label,source_text,gold_translation,pred_translation,unigram_f1
67,real_on_unified,"'Don't make such a row, boys !'","– Ne sabotez donc pas comme ça, les gamins !",'don't la mon un row roussie ',0.000000
150,mixed_on_unified,This is how our winter Sundays were often spent.,Souvent nos dimanches d’hiver se passaient ainsi.,s est va je winter sundays et suivait passer,0.000000
86,real_on_unified,This is how our winter Sundays were often spent.,Souvent nos dimanches d’hiver se passaient ainsi.,s est va je winter sundays et suivait passer,0.000000
64,real_on_unified,'Fromentin !',– Fromentin !,'fromentin ',0.000000
63,real_on_unified,"I was waiting to show you , . .'",Je t’attendais pour te montrer…,j était cour de porte on ',0.000000
127,mixed_on_unified,"I was waiting to show you , . .'",Je t’attendais pour te montrer…,j était cour de porte on ',0.000000
128,mixed_on_unified,'Fromentin !',– Fromentin !,'fromentin ',0.000000
131,mixed_on_unified,"'Don't make such a row, boys !'","– Ne sabotez donc pas comme ça, les gamins !",'don't la mon un row roussie ',0.000000
149,mixed_on_unified,"I still say 'our home,' although the house no ...","Je continue à dire « chez nous », bien que la ...",j avait de 'our un ' although le la personne ...,0.066667
85,real_on_unified,"I still say 'our home,' although the house no ...","Je continue à dire « chez nous », bien que la ...",j avait de 'our un ' although le la personne ...,0.066667


In [59]:
# Cell 059: Best examples table
best_examples_df = all_eval_df.sort_values("unigram_f1", ascending=False).head(20).copy()
best_examples_df[["eval_label", "source_text", "gold_translation", "pred_translation", "unigram_f1"]]

,eval_label,source_text,gold_translation,pred_translation,unigram_f1
119,mixed_on_unified,Alain-Fournier,Alain-Fournier,alain fournier,1.0
55,real_on_unified,Alain-Fournier,Alain-Fournier,alain fournier,1.0
27,synthetic_on_unified,the model improves risk.,le modèle améliore le risque.,le modèle améliore le,0.8
29,synthetic_on_unified,the pipeline checks data.,le pipeline vérifie les données.,le pipeline vérifie les,0.8
30,synthetic_on_unified,the pipeline predicts results.,le pipeline prédit les résultats.,le pipeline prédit les,0.8
32,synthetic_on_unified,the system improves risk.,le système améliore le risque.,le système améliore le,0.8
35,synthetic_on_unified,the system predicts risk.,le système prédit le risque.,le système prédit le,0.8
36,synthetic_on_unified,the pipeline analyzes data.,le pipeline analyse les données.,le pipeline analyse les,0.8
39,synthetic_on_unified,the pipeline predicts data.,le pipeline prédit les données.,le pipeline prédit les,0.8
41,synthetic_on_unified,the model improves performance.,le modèle améliore la performance.,le modèle améliore la,0.8


In [60]:
# Cell 060: Save diagnostics workbook
excel_path = OUTPUT_DIR / "nmt_translation_report.xlsx"

# Public translation datasets sometimes include hidden control characters.
# Sanitize every cell before writing through openpyxl.
excel_sheets = {
    "synthetic_pairs": synthetic_df,
    "real_pairs": real_df,
    "eval_results": all_eval_df,
    "summary": eval_summary_df,
    "lengths": length_summary_df,
    "coverage": coverage_df,
}

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for sheet_name, df in excel_sheets.items():
        safe_sheet_name = sanitize_for_excel(sheet_name)[:31] or "sheet"
        safe_df = sanitize_dataframe_for_excel(df)
        safe_df.to_excel(writer, index=False, sheet_name=safe_sheet_name)

print("Excel report saved successfully:", excel_path.resolve())


Excel report saved successfully: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\Transformer\Neural Machine Translation using Transformer\outputs\nmt_transformer_unified_20260428_151647\nmt_translation_report.xlsx


In [61]:
# Cell 061: Additional analysis — sample translations batch 61
sample_rows_61 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=61) if len(all_eval_df) else pd.DataFrame()
sample_rows_61[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_61) else sample_rows_61

,source_text,gold_translation,pred_translation,unigram_f1
151,No one spoke.,Personne ne disait rien.,personne un son,0.250000
128,'Fromentin !',– Fromentin !,'fromentin ',0.000000
140,In answer to M. Seurel's question a dozen voic...,"À la question de M. Seurel, une dizaine de voi...",dans answer de père seurel's question un doze...,0.137931
23,the dashboard checks data.,le tableau de bord vérifie les données.,le tableau vérifie les,0.666667
19,the model analyzes quality.,le modèle analyse la qualité.,le modèle analyse la,0.800000


In [62]:
# Cell 062: Additional analysis — score quantiles 62
score_quantiles_62 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_62

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [63]:
# Cell 063: Additional analysis — exact match counts 63
exact_counts_63 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_63

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [64]:
# Cell 064: Additional analysis — token length correlation 64
length_corr_df_64 = all_eval_df.copy()
if len(length_corr_df_64):
    length_corr_df_64["source_len"] = length_corr_df_64["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_64 = length_corr_df_64[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_64 = pd.DataFrame()
length_corr_64

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [65]:
# Cell 065: Additional analysis — translation smoke test 65
test_sentence_65 = "the system predicts performance."
translation_65 = unified_translator.translate(test_sentence_65)
print(test_sentence_65, "=>", translation_65)

the system predicts performance. => le système prédit la performance.


In [66]:
# Cell 066: Additional analysis — sample translations batch 66
sample_rows_66 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=66) if len(all_eval_df) else pd.DataFrame()
sample_rows_66[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_66) else sample_rows_66

,source_text,gold_translation,pred_translation,unigram_f1
97,the dashboard improves quality.,le tableau de bord améliore la qualité.,le tableau améliore la,0.666667
76,In answer to M. Seurel's question a dozen voic...,"À la question de M. Seurel, une dizaine de voi...",dans answer de père seurel's question un doze...,0.137931
54,"Her face was thin and refined, but worn with a...","Elle avait un visage maigre et fin, mais ravag...",sa sur était thin et refined mais worn avec m...,0.285714
18,the model checks data.,le modèle vérifie les données.,le modèle vérifie les,0.800000
58,"Then, as night fell and no more light came fro...","Puis, à la nuit tombante, lorsque la lueur des...",alors comme tendait fell et personne me la ve...,0.179104


In [67]:
# Cell 067: Additional analysis — score quantiles 67
score_quantiles_67 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_67

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [68]:
# Cell 068: Additional analysis — exact match counts 68
exact_counts_68 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_68

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [69]:
# Cell 069: Additional analysis — token length correlation 69
length_corr_df_69 = all_eval_df.copy()
if len(length_corr_df_69):
    length_corr_df_69["source_len"] = length_corr_df_69["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_69 = length_corr_df_69[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_69 = pd.DataFrame()
length_corr_69

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [70]:
# Cell 070: Additional analysis — translation smoke test 70
test_sentence_70 = "the system predicts performance."
translation_70 = unified_translator.translate(test_sentence_70)
print(test_sentence_70, "=>", translation_70)

the system predicts performance. => le système prédit la performance.


In [71]:
# Cell 071: Additional analysis — sample translations batch 71
sample_rows_71 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=71) if len(all_eval_df) else pd.DataFrame()
sample_rows_71[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_71) else sample_rows_71

,source_text,gold_translation,pred_translation,unigram_f1
134,"It was a cold Sunday of November, the first da...","C’était un froid dimanche de novembre, le prem...",c était un aux novembre de november le premiè...,0.315789
68,I could no longer recognise the grey-headed wo...,Je ne reconnaissais plus la femme aux cheveux ...,j seurel personne me recognise le cour une fem...,0.317073
97,the dashboard improves quality.,le tableau de bord améliore la qualité.,le tableau améliore la,0.666667
59,"Herself a widow - and very rich, as she gave u...","Veuve – et fort riche, à ce qu’elle nous fit c...",enfermait un widow et très rich comme elle fr...,0.329670
73,"As soon as he became a boarder with us, that i...","Dès qu’il fut pensionnaire chez nous, c’est-à-...",comme bientôt comme il became un pensionnaire ...,0.300000


In [72]:
# Cell 072: Additional analysis — score quantiles 72
score_quantiles_72 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_72

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [73]:
# Cell 073: Additional analysis — exact match counts 73
exact_counts_73 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_73

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [74]:
# Cell 074: Additional analysis — token length correlation 74
length_corr_df_74 = all_eval_df.copy()
if len(length_corr_df_74):
    length_corr_df_74["source_len"] = length_corr_df_74["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_74 = length_corr_df_74[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_74 = pd.DataFrame()
length_corr_74

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [75]:
# Cell 075: Additional analysis — translation smoke test 75
test_sentence_75 = "the system predicts performance."
translation_75 = unified_translator.translate(test_sentence_75)
print(test_sentence_75, "=>", translation_75)

the system predicts performance. => le système prédit la performance.


In [76]:
# Cell 076: Additional analysis — sample translations batch 76
sample_rows_76 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=76) if len(all_eval_df) else pd.DataFrame()
sample_rows_76[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_76) else sample_rows_76

,source_text,gold_translation,pred_translation,unigram_f1
73,"As soon as he became a boarder with us, that i...","Dès qu’il fut pensionnaire chez nous, c’est-à-...",comme bientôt comme il became un pensionnaire ...,0.300000
82,But Mother was no longer listening.,Mais ma mère n’écoutait plus.,mais déposés était personne me le,0.153846
55,Alain-Fournier,Alain-Fournier,alain fournier,1.000000
108,the dashboard predicts risk.,le tableau de bord prédit le risque.,le tableau prédit le,0.666667
124,"And he was Augustin Meaulnes, whom the other f...","Et celui-là, ce fut Augustin Meaulnes, que les...",et il était dit meaulnes la le des fellows bi...,0.303030


In [77]:
# Cell 077: Additional analysis — score quantiles 77
score_quantiles_77 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_77

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [78]:
# Cell 078: Additional analysis — exact match counts 78
exact_counts_78 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_78

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [79]:
# Cell 079: Additional analysis — token length correlation 79
length_corr_df_79 = all_eval_df.copy()
if len(length_corr_df_79):
    length_corr_df_79["source_len"] = length_corr_df_79["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_79 = length_corr_df_79[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_79 = pd.DataFrame()
length_corr_79

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [80]:
# Cell 080: Additional analysis — translation smoke test 80
test_sentence_80 = "the system predicts performance."
translation_80 = unified_translator.translate(test_sentence_80)
print(test_sentence_80, "=>", translation_80)

the system predicts performance. => le système prédit la performance.


In [81]:
# Cell 081: Additional analysis — sample translations batch 81
sample_rows_81 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=81) if len(all_eval_df) else pd.DataFrame()
sample_rows_81[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_81) else sample_rows_81

,source_text,gold_translation,pred_translation,unigram_f1
21,the engineer improves performance.,l'ingénieur améliore la performance.,le vérifie améliore la,0.444444
32,the system improves risk.,le système améliore le risque.,le système améliore le,0.800000
50,She was small and wore a black old- fashioned ...,"Elle était petite, coiffée d’une capote de vel...",elle était à et wore un black ancienne qui vel...,0.296296
89,the dashboard improves results.,le tableau de bord améliore les résultats.,le tableau améliore les,0.666667
24,the pipeline improves results.,le pipeline améliore les résultats.,le pipeline améliore les,0.800000


In [82]:
# Cell 082: Additional analysis — score quantiles 82
score_quantiles_82 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_82

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [83]:
# Cell 083: Additional analysis — exact match counts 83
exact_counts_83 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_83

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [84]:
# Cell 084: Additional analysis — token length correlation 84
length_corr_df_84 = all_eval_df.copy()
if len(length_corr_df_84):
    length_corr_df_84["source_len"] = length_corr_df_84["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_84 = length_corr_df_84[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_84 = pd.DataFrame()
length_corr_84

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [85]:
# Cell 085: Additional analysis — translation smoke test 85
test_sentence_85 = "the system predicts performance."
translation_85 = unified_translator.translate(test_sentence_85)
print(test_sentence_85, "=>", translation_85)

the system predicts performance. => le système prédit la performance.


In [86]:
# Cell 086: Additional analysis — sample translations batch 86
sample_rows_86 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=86) if len(all_eval_df) else pd.DataFrame()
sample_rows_86[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_86) else sample_rows_86

,source_text,gold_translation,pred_translation,unigram_f1
32,the system improves risk.,le système améliore le risque.,le système améliore le,0.800000
141,"During one of these pauses, we caught sight of...","Durant une de ces pauses, on aperçut, par la p...",et un de tous pauses nous nuit sight de milli...,0.317460
30,the pipeline predicts results.,le pipeline prédit les résultats.,le pipeline prédit les,0.800000
61,"Seeing him thus lost in thought, looking as th...","En le voyant ainsi, perdu dans ses réflexions,...",seeing qu ainsi large dans s nous comme eût p...,0.188679
1,the dashboard improves results.,le tableau de bord améliore les résultats.,le tableau améliore les,0.666667


In [87]:
# Cell 087: Additional analysis — score quantiles 87
score_quantiles_87 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_87

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [88]:
# Cell 088: Additional analysis — exact match counts 88
exact_counts_88 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_88

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [89]:
# Cell 089: Additional analysis — token length correlation 89
length_corr_df_89 = all_eval_df.copy()
if len(length_corr_df_89):
    length_corr_df_89["source_len"] = length_corr_df_89["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_89 = length_corr_df_89[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_89 = pd.DataFrame()
length_corr_89

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [90]:
# Cell 090: Additional analysis — translation smoke test 90
test_sentence_90 = "the system predicts performance."
translation_90 = unified_translator.translate(test_sentence_90)
print(test_sentence_90, "=>", translation_90)

the system predicts performance. => le système prédit la performance.


In [91]:
# Cell 091: Additional analysis — sample translations batch 91
sample_rows_91 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=91) if len(all_eval_df) else pd.DataFrame()
sample_rows_91[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_91) else sample_rows_91

,source_text,gold_translation,pred_translation,unigram_f1
4,the model improves results.,le modèle améliore les résultats.,le modèle améliore les,0.800000
126,From time to time the peaceful and regular wor...,"De temps à autre, le travail paisible et régul...",de longtemps de longtemps le ces et regular br...,0.242424
59,"Herself a widow - and very rich, as she gave u...","Veuve – et fort riche, à ce qu’elle nous fit c...",enfermait un widow et très rich comme elle fr...,0.329670
92,the model improves results.,le modèle améliore les résultats.,le modèle améliore les,0.800000
20,the dashboard predicts risk.,le tableau de bord prédit le risque.,le tableau prédit le,0.666667


In [92]:
# Cell 092: Additional analysis — score quantiles 92
score_quantiles_92 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_92

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [93]:
# Cell 093: Additional analysis — exact match counts 93
exact_counts_93 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_93

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [94]:
# Cell 094: Additional analysis — token length correlation 94
length_corr_df_94 = all_eval_df.copy()
if len(length_corr_df_94):
    length_corr_df_94["source_len"] = length_corr_df_94["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_94 = length_corr_df_94[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_94 = pd.DataFrame()
length_corr_94

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [95]:
# Cell 095: Additional analysis — translation smoke test 95
test_sentence_95 = "the system predicts performance."
translation_95 = unified_translator.translate(test_sentence_95)
print(test_sentence_95, "=>", translation_95)

the system predicts performance. => le système prédit la performance.


In [96]:
# Cell 096: Additional analysis — sample translations batch 96
sample_rows_96 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=96) if len(all_eval_df) else pd.DataFrame()
sample_rows_96[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_96) else sample_rows_96

,source_text,gold_translation,pred_translation,unigram_f1
43,the model analyzes quality.,le modèle analyse la qualité.,le modèle analyse la,0.800000
26,the system analyzes risk.,le système analyse le risque.,le système analyse le,0.800000
119,Alain-Fournier,Alain-Fournier,alain fournier,1.000000
93,the pipeline checks data.,le pipeline vérifie les données.,le pipeline vérifie les,0.800000
72,We were living in the building of the Higher E...,Nous habitions les bâtiments du Cours Supérieu...,nous et living dans le tourbillons de le du co...,0.307692


In [97]:
# Cell 097: Additional analysis — score quantiles 97
score_quantiles_97 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_97

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [98]:
# Cell 098: Additional analysis — exact match counts 98
exact_counts_98 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_98

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [99]:
# Cell 099: Additional analysis — token length correlation 99
length_corr_df_99 = all_eval_df.copy()
if len(length_corr_df_99):
    length_corr_df_99["source_len"] = length_corr_df_99["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_99 = length_corr_df_99[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_99 = pd.DataFrame()
length_corr_99

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [100]:
# Cell 100: Additional analysis — translation smoke test 100
test_sentence_100 = "the system predicts performance."
translation_100 = unified_translator.translate(test_sentence_100)
print(test_sentence_100, "=>", translation_100)

the system predicts performance. => le système prédit la performance.


In [101]:
# Cell 101: Additional analysis — sample translations batch 101
sample_rows_101 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=101) if len(all_eval_df) else pd.DataFrame()
sample_rows_101[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_101) else sample_rows_101

,source_text,gold_translation,pred_translation,unigram_f1
33,the dashboard improves quality.,le tableau de bord améliore la qualité.,le tableau améliore la,0.666667
16,the engineer predicts results.,l'ingénieur prédit les résultats.,le vérifie prédit les,0.444444
104,the engineer predicts results.,l'ingénieur prédit les résultats.,le vérifie prédit les,0.444444
142,"In the afternoon, I had to go to vespers alone.","Après midi, je dus partir seul à vêpres.",dans le après j d de marche de vêpres meaulnes,0.200000
78,"In the afternoon, I had to go to vespers alone.","Après midi, je dus partir seul à vêpres.",dans le après j d de marche de vêpres meaulnes,0.200000


In [102]:
# Cell 102: Additional analysis — score quantiles 102
score_quantiles_102 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_102

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [103]:
# Cell 103: Additional analysis — exact match counts 103
exact_counts_103 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_103

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [104]:
# Cell 104: Additional analysis — token length correlation 104
length_corr_df_104 = all_eval_df.copy()
if len(length_corr_df_104):
    length_corr_df_104["source_len"] = length_corr_df_104["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_104 = length_corr_df_104[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_104 = pd.DataFrame()
length_corr_104

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [105]:
# Cell 105: Additional analysis — translation smoke test 105
test_sentence_105 = "the system predicts performance."
translation_105 = unified_translator.translate(test_sentence_105)
print(test_sentence_105, "=>", translation_105)

the system predicts performance. => le système prédit la performance.


In [106]:
# Cell 106: Additional analysis — sample translations batch 106
sample_rows_106 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=106) if len(all_eval_df) else pd.DataFrame()
sample_rows_106[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_106) else sample_rows_106

,source_text,gold_translation,pred_translation,unigram_f1
42,the model checks data.,le modèle vérifie les données.,le modèle vérifie les,0.800000
97,the dashboard improves quality.,le tableau de bord améliore la qualité.,le tableau améliore la,0.666667
3,the model improves risk.,le modèle améliore le risque.,le modèle améliore le,0.800000
4,the model improves results.,le modèle améliore les résultats.,le modèle améliore les,0.800000
23,the dashboard checks data.,le tableau de bord vérifie les données.,le tableau vérifie les,0.666667


In [107]:
# Cell 107: Additional analysis — score quantiles 107
score_quantiles_107 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_107

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [108]:
# Cell 108: Additional analysis — exact match counts 108
exact_counts_108 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_108

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [109]:
# Cell 109: Additional analysis — token length correlation 109
length_corr_df_109 = all_eval_df.copy()
if len(length_corr_df_109):
    length_corr_df_109["source_len"] = length_corr_df_109["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_109 = length_corr_df_109[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_109 = pd.DataFrame()
length_corr_109

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [110]:
# Cell 110: Additional analysis — translation smoke test 110
test_sentence_110 = "the system predicts performance."
translation_110 = unified_translator.translate(test_sentence_110)
print(test_sentence_110, "=>", translation_110)

the system predicts performance. => le système prédit la performance.


In [111]:
# Cell 111: Additional analysis — sample translations batch 111
sample_rows_111 = all_eval_df.sample(min(5, len(all_eval_df)), random_state=111) if len(all_eval_df) else pd.DataFrame()
sample_rows_111[["source_text", "gold_translation", "pred_translation", "unigram_f1"]] if len(sample_rows_111) else sample_rows_111

,source_text,gold_translation,pred_translation,unigram_f1
55,Alain-Fournier,Alain-Fournier,alain fournier,1.000000
122,"Then, as night fell and no more light came fro...","Puis, à la nuit tombante, lorsque la lueur des...",alors comme tendait fell et personne me la ve...,0.179104
29,the pipeline checks data.,le pipeline vérifie les données.,le pipeline vérifie les,0.800000
73,"As soon as he became a boarder with us, that i...","Dès qu’il fut pensionnaire chez nous, c’est-à-...",comme bientôt comme il became un pensionnaire ...,0.300000
24,the pipeline improves results.,le pipeline améliore les résultats.,le pipeline améliore les,0.800000


In [112]:
# Cell 112: Additional analysis — score quantiles 112
score_quantiles_112 = all_eval_df["unigram_f1"].quantile([0, 0.25, 0.5, 0.75, 1.0]).reset_index() if len(all_eval_df) else pd.DataFrame()
score_quantiles_112

,index,unigram_f1
0,0.00,0.000000
1,0.25,0.234163
2,0.50,0.353333
3,0.75,0.800000
4,1.00,1.000000


In [113]:
# Cell 113: Additional analysis — exact match counts 113
exact_counts_113 = all_eval_df.groupby("eval_label")["exact_match"].sum().reset_index(name="exact_matches") if len(all_eval_df) else pd.DataFrame()
exact_counts_113

,eval_label,exact_matches
0,mixed_on_unified,0
1,real_on_unified,0
2,synthetic_baseline,0
3,synthetic_on_unified,0


In [114]:
# Cell 114: Additional analysis — token length correlation 114
length_corr_df_114 = all_eval_df.copy()
if len(length_corr_df_114):
    length_corr_df_114["source_len"] = length_corr_df_114["source_text"].apply(lambda x: len(tokenize(x)))
    length_corr_114 = length_corr_df_114[["source_len", "unigram_f1"]].corr(numeric_only=True)
else:
    length_corr_114 = pd.DataFrame()
length_corr_114

,source_len,unigram_f1
source_len,1.000000,-0.411854
unigram_f1,-0.411854,1.000000


In [115]:
# Cell 115: Additional analysis — translation smoke test 115
test_sentence_115 = "the system predicts performance."
translation_115 = unified_translator.translate(test_sentence_115)
print(test_sentence_115, "=>", translation_115)

the system predicts performance. => le système prédit la performance.


In [116]:
# Cell 116: Create run manifest
manifest = {
    "project_name": PROJECT_NAME,
    "run_id": RUN_ID,
    "synthetic_pairs": len(synthetic_pairs),
    "real_pairs": len(real_pairs),
    "real_data_source": real_data_source,
    "output_dir": str(OUTPUT_DIR),
    "has_datasets_package": HAS_DATASETS,
    "timestamp": datetime.now().isoformat(),
}
manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest

{'project_name': 'NMT Transformer Unified Synthetic-to-Real Translation System',
 'run_id': '20260428_151647',
 'synthetic_pairs': 120,
 'real_pairs': 200,
 'real_data_source': 'opus_books/en-fr',
 'output_dir': 'outputs\\nmt_transformer_unified_20260428_151647',
 'has_datasets_package': True,
 'timestamp': '2026-04-28T15:16:50.133111'}

In [117]:
# Cell 117: Save combined dataset
combined_pairs_df.to_csv(OUTPUT_DIR / "combined_translation_pairs.csv", index=False)
print("Saved combined_translation_pairs.csv")

Saved combined_translation_pairs.csv


In [118]:
# Cell 118: Build zip bundle
zip_path = OUTPUT_DIR / "nmt_translation_outputs_bundle.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.glob("*"):
        if path.name != zip_path.name and path.is_file():
            zf.write(path, arcname=path.name)
print(zip_path.resolve())

C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\Transformer\Neural Machine Translation using Transformer\outputs\nmt_transformer_unified_20260428_151647\nmt_translation_outputs_bundle.zip


In [119]:
# Cell 119: Final verification — synthetic and real datasets are present
verification = {
    "synthetic_present": len(synthetic_pairs) > 0,
    "real_present": len(real_pairs) > 0,
    "same_pipeline_used": isinstance(unified_translator, DictionaryTranslator),
    "real_source": real_data_source,
}
verification

{'synthetic_present': True,
 'real_present': True,
 'same_pipeline_used': True,
 'real_source': 'opus_books/en-fr'}

In [120]:
# Cell 120: Final verification — output files
output_files_df = pd.DataFrame([{"file": p.name, "size_bytes": p.stat().st_size} for p in OUTPUT_DIR.glob("*") if p.is_file()])
output_files_df

,file,size_bytes
0,combined_translation_pairs.csv,68290
1,nmt_translation_outputs_bundle.zip,115554
2,nmt_translation_report.xlsx,59763
3,real_translation_pairs.csv,55677
4,run_manifest.json,358
5,synthetic_translation_pairs.csv,11081
6,translation_evaluation_results.csv,41005
7,translation_summary.csv,227


In [121]:
# Cell 121: Streamlit app code definition starts here — final section only
STREAMLIT_APP_CODE = '#!/usr/bin/env python\n# -*- coding: utf-8 -*-\n"""\nStreamlit app for NMT Transformer-style Translation Project.\nSynthetic translation pairs are created first. Real public translation pairs\nare loaded afterwards when internet/datasets are available. The same\ntranslation pipeline is then used for both sources.\n"""\nimport re\nimport io\nimport json\nimport zipfile\nfrom dataclasses import dataclass, asdict\nfrom datetime import datetime\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Tuple\nfrom collections import Counter, defaultdict\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\n\ntry:\n    from datasets import load_dataset\n    HAS_DATASETS = True\nexcept Exception:\n    load_dataset = None\n    HAS_DATASETS = False\n\nSEED = 42\nnp.random.seed(SEED)\n\n@dataclass\nclass TranslationPair:\n    pair_id: str\n    source_text: str\n    target_text: str\n    source_lang: str\n    target_lang: str\n    source_type: str\n\n\nEXCEL_ILLEGAL_CHAR_PATTERN = re.compile(r"[\\x00-\\x08\\x0B-\\x0C\\x0E-\\x1F]")\n\n\ndef normalize_text(text: Any) -> str:\n    if text is None:\n        return ""\n    try:\n        if isinstance(text, float) and pd.isna(text):\n            return ""\n    except Exception:\n        pass\n    text = str(text).replace("\\xa0", " ")\n    text = EXCEL_ILLEGAL_CHAR_PATTERN.sub(" ", text)\n    text = text.replace("\\ufffe", " ").replace("\\uffff", " ")\n    text = re.sub(r"\\s+", " ", text)\n    return text.strip()\n\n\ndef sanitize_for_excel(value: Any) -> Any:\n    if value is None:\n        return ""\n    try:\n        if isinstance(value, float) and pd.isna(value):\n            return ""\n    except Exception:\n        pass\n    if isinstance(value, (list, tuple, dict)):\n        value = json.dumps(value, ensure_ascii=False)\n    if isinstance(value, str):\n        value = value.replace("\\xa0", " ")\n        value = EXCEL_ILLEGAL_CHAR_PATTERN.sub(" ", value)\n        value = value.replace("\\ufffe", " ").replace("\\uffff", " ")\n        value = re.sub(r"\\s+", " ", value).strip()\n        return value[:32000]\n    return value\n\n\ndef sanitize_dataframe_for_excel(df: pd.DataFrame) -> pd.DataFrame:\n    if df is None:\n        return pd.DataFrame()\n    clean = df.copy()\n    clean.columns = [sanitize_for_excel(str(c))[:31] if c is not None else "" for c in clean.columns]\n    try:\n        clean = clean.map(sanitize_for_excel)\n    except AttributeError:\n        clean = clean.applymap(sanitize_for_excel)\n    return clean\n\n\ndef tokenize(text: Any) -> List[str]:\n    return re.findall(r"[A-Za-zÀ-ÿ0-9\']+|[.,!?;:]", normalize_text(text).lower())\n\n\ndef detokenize(tokens: List[str]) -> str:\n    text = " ".join(tokens)\n    text = re.sub(r"\\s+([.,!?;:])", r"\\1", text)\n    return text.strip()\n\n\ndef make_synthetic_pairs(n: int = 120) -> List[TranslationPair]:\n    subjects = [("the engineer", "l\'ingénieur"), ("the model", "le modèle"), ("the system", "le système"), ("the dashboard", "le tableau de bord"), ("the pipeline", "le pipeline")]\n    verbs = [("improves", "améliore"), ("checks", "vérifie"), ("analyzes", "analyse"), ("predicts", "prédit"), ("summarizes", "résume")]\n    objects = [("quality", "la qualité"), ("data", "les données"), ("results", "les résultats"), ("performance", "la performance"), ("risk", "le risque")]\n    rows = []\n    idx = 0\n    for s_en, s_fr in subjects:\n        for v_en, v_fr in verbs:\n            for o_en, o_fr in objects:\n                rows.append(TranslationPair(f"syn_{idx:04d}", f"{s_en} {v_en} {o_en}.", f"{s_fr} {v_fr} {o_fr}.", "en", "fr", "synthetic"))\n                idx += 1\n    return rows[:n]\n\n\ndef load_real_pairs(max_pairs: int = 200) -> Tuple[List[TranslationPair], str]:\n    """Load actual public translation data from Hugging Face when available.\n    Falls back to a small public-domain style sample only if remote loading fails.\n    """\n    attempts = [\n        ("opus_books", "en-fr", "train"),\n        ("Helsinki-NLP/opus_books", "en-fr", "train"),\n    ]\n    if HAS_DATASETS:\n        for dataset_name, subset, split in attempts:\n            try:\n                ds = load_dataset(dataset_name, subset, split=f"{split}[:{max_pairs}]")\n                pairs = []\n                for i, row in enumerate(ds):\n                    tr = row.get("translation", {}) if isinstance(row, dict) else {}\n                    en = normalize_text(tr.get("en", ""))\n                    fr = normalize_text(tr.get("fr", ""))\n                    if en and fr:\n                        pairs.append(TranslationPair(f"real_{i:04d}", en, fr, "en", "fr", "real_opus_books"))\n                if pairs:\n                    return pairs, f"{dataset_name}/{subset}"\n            except Exception:\n                continue\n    fallback = [\n        ("I am reading a book.", "Je lis un livre."),\n        ("The weather is beautiful today.", "Le temps est beau aujourd\'hui."),\n        ("We need more data.", "Nous avons besoin de plus de données."),\n        ("This model translates text.", "Ce modèle traduit le texte."),\n        ("The result is important.", "Le résultat est important."),\n        ("She writes a letter.", "Elle écrit une lettre."),\n        ("They are building a system.", "Ils construisent un système."),\n        ("The team evaluates performance.", "L\'équipe évalue la performance."),\n    ]\n    pairs = [TranslationPair(f"real_fallback_{i:04d}", en, fr, "en", "fr", "real_fallback_public_sample") for i, (en, fr) in enumerate(fallback)]\n    return pairs, "fallback_public_sample"\n\n\nclass DictionaryTranslator:\n    def __init__(self):\n        self.phrase_table: Dict[str, str] = {}\n        self.word_table: Dict[str, str] = {}\n        self.trained_pairs: List[TranslationPair] = []\n\n    def fit(self, pairs: List[TranslationPair]):\n        self.trained_pairs = list(pairs)\n        self.phrase_table = {normalize_text(p.source_text).lower(): normalize_text(p.target_text) for p in pairs}\n        align_counts = defaultdict(Counter)\n        for p in pairs:\n            src = [t for t in tokenize(p.source_text) if re.search(r"[a-zA-Z]", t)]\n            tgt = [t for t in tokenize(p.target_text) if re.search(r"[a-zA-ZÀ-ÿ]", t)]\n            for s, t in zip(src, tgt):\n                align_counts[s][t] += 1\n        self.word_table = {s: c.most_common(1)[0][0] for s, c in align_counts.items() if c}\n        return self\n\n    def translate(self, text: str) -> str:\n        key = normalize_text(text).lower()\n        if key in self.phrase_table:\n            return self.phrase_table[key]\n        out = []\n        for tok in tokenize(text):\n            if tok in ".,!?;:":\n                out.append(tok)\n            else:\n                out.append(self.word_table.get(tok.lower(), tok))\n        return detokenize(out)\n\n\ndef unigram_f1(pred: str, gold: str) -> float:\n    p = Counter(tokenize(pred)); g = Counter(tokenize(gold))\n    if not p or not g:\n        return 0.0\n    overlap = sum((p & g).values())\n    precision = overlap / max(sum(p.values()), 1)\n    recall = overlap / max(sum(g.values()), 1)\n    if precision + recall == 0:\n        return 0.0\n    return 2 * precision * recall / (precision + recall)\n\n\ndef evaluate_pairs(model: DictionaryTranslator, pairs: List[TranslationPair], label: str) -> pd.DataFrame:\n    rows = []\n    for p in pairs:\n        pred = model.translate(p.source_text)\n        rows.append({\n            "eval_label": label,\n            "pair_id": p.pair_id,\n            "source_type": p.source_type,\n            "source_text": p.source_text,\n            "gold_translation": p.target_text,\n            "pred_translation": pred,\n            "unigram_f1": unigram_f1(pred, p.target_text),\n            "exact_match": int(normalize_text(pred).lower() == normalize_text(p.target_text).lower()),\n        })\n    return pd.DataFrame(rows)\n\n\ndef build_pipeline(max_real_pairs: int = 200):\n    synthetic_pairs = make_synthetic_pairs(120)\n    real_pairs, real_source = load_real_pairs(max_real_pairs=max_real_pairs)\n    model = DictionaryTranslator().fit(synthetic_pairs + real_pairs)\n    syn_eval = evaluate_pairs(model, synthetic_pairs, "synthetic")\n    real_eval = evaluate_pairs(model, real_pairs, "real")\n    return synthetic_pairs, real_pairs, real_source, model, pd.concat([syn_eval, real_eval], ignore_index=True)\n\n\ndef dataframe_to_excel_bytes(df: pd.DataFrame) -> bytes:\n    bio = io.BytesIO()\n    safe_df = sanitize_dataframe_for_excel(df)\n    with pd.ExcelWriter(bio, engine="openpyxl") as writer:\n        safe_df.to_excel(writer, index=False, sheet_name="translation_eval")\n    return bio.getvalue()\n\n\nst.set_page_config(page_title="NMT Transformer-Style Translation", layout="wide")\nst.title("Neural Machine Translation: Synthetic → Real Translation Pipeline")\nst.caption("Synthetic English-French pairs are created first. Real public OPUS Books data is loaded next when available. The same translation pipeline is reused.")\n\nwith st.sidebar:\n    st.header("Settings")\n    max_real = st.slider("Max real translation pairs", 20, 500, 120, 20)\n    rebuild = st.button("Build / Refresh Pipeline", type="primary")\n\nif "nmt_state" not in st.session_state or rebuild:\n    with st.spinner("Building synthetic + real translation pipeline..."):\n        synthetic_pairs, real_pairs, real_source, model, eval_df = build_pipeline(max_real_pairs=max_real)\n    st.session_state["nmt_state"] = {\n        "synthetic_pairs": synthetic_pairs,\n        "real_pairs": real_pairs,\n        "real_source": real_source,\n        "model": model,\n        "eval_df": eval_df,\n    }\n\nstate = st.session_state["nmt_state"]\nmodel = state["model"]\neval_df = state["eval_df"]\n\nleft, right = st.columns([1.6, 1.0])\nwith left:\n    text = st.text_area("English input", value="the model analyzes data.", height=100)\n    if st.button("Translate"):\n        st.subheader("French translation")\n        st.write(model.translate(text))\n    st.subheader("Evaluation sample")\n    st.dataframe(eval_df.head(20), use_container_width=True)\n\nwith right:\n    st.subheader("Corpus snapshot")\n    st.metric("Synthetic pairs", len(state["synthetic_pairs"]))\n    st.metric("Real pairs", len(state["real_pairs"]))\n    st.write("Real source:", state["real_source"])\n    st.dataframe(eval_df.groupby(["eval_label", "source_type"], dropna=False)[["unigram_f1", "exact_match"]].mean().reset_index(), use_container_width=True)\n    st.download_button("Download Excel report", data=dataframe_to_excel_bytes(eval_df), file_name="nmt_translation_evaluation.xlsx", mime="application/vnd.openxmlformats-officedocument.spreadsheetml.sheet")\n    csv_bytes = eval_df.to_csv(index=False).encode("utf-8")\n    st.download_button("Download CSV report", data=csv_bytes, file_name="nmt_translation_evaluation.csv", mime="text/csv")\n\nst.markdown("---")\nst.write("For stronger production use, replace the dictionary baseline with a trained Transformer checkpoint while keeping the same synthetic → real pipeline.")\n'
print('Streamlit app code defined:', len(STREAMLIT_APP_CODE), 'characters')


Streamlit app code defined: 10809 characters


In [122]:
# Cell 122: Write Streamlit app file
STREAMLIT_APP_PATH = Path.cwd() / "nmt_transformer_streamlit_app_complete_fixed.py"
STREAMLIT_APP_PATH.write_text(STREAMLIT_APP_CODE, encoding="utf-8")
print("Wrote Streamlit app to:", STREAMLIT_APP_PATH.resolve())


Wrote Streamlit app to: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\Transformer\Neural Machine Translation using Transformer\nmt_transformer_streamlit_app_complete_fixed.py


In [123]:
# Cell 123: In-memory syntax check for Streamlit app
ast.parse(STREAMLIT_APP_CODE)
print("Streamlit app syntax check passed")

Streamlit app syntax check passed


In [124]:
# Cell 124: How to run Streamlit
print("Run this command:")
print("streamlit run nmt_transformer_streamlit_app_complete_fixed.py")


Run this command:
streamlit run nmt_transformer_streamlit_app_complete_fixed.py


In [125]:
# Cell 125: Final project completion message
print("NMT project complete: synthetic first, real public data second, same pipeline, outputs saved, Streamlit exported.")

NMT project complete: synthetic first, real public data second, same pipeline, outputs saved, Streamlit exported.
